Надо построить аберрационную траекторию звезды (ее положение задается RA,DEC) на протяжении года в экваториальных координатах

In [72]:
from math import *
import numpy as np
from datetime import datetime
from astropy.time import Time
from astroquery.jplhorizons import Horizons

import astropy.units as u
import astropy.constants as const
from astropy.coordinates import SkyCoord

In [11]:
def tangFromRADE(ra, dec, RA, DEC):
    ksi = cos(dec)*sin(ra-RA)/(sin(dec)*sin(DEC)+cos(dec)*cos(DEC)*cos(ra-RA))
    eta = (sin(dec)*cos(DEC)-cos(dec)*sin(DEC)*cos(ra-RA))/(sin(dec)*sin(DEC)+cos(dec)*cos(DEC)*cos(ra-RA))
    return ksi,eta

In [141]:
# эту функцию предстоит реализовать самостоятельно
def aberration(ra,dec,e_dot_x,e_dot_y,e_dot_z):
    coord = SkyCoord(ra=ra*u.rad, dec=dec*u.rad)
    e_coord = SkyCoord(x=e_dot_x, y=e_dot_y, z=e_dot_z, frame='icrs', representation_type='cartesian')
    r = coord.cartesian
    e = e_coord.cartesian
    
    theta = coord.separation(e_coord)
    
    v = np.linalg.norm(e.xyz)
    c = const.c.to(u.au/u.day).value # c в тех же единицах что и v
    B =  v / c
    Gamma = np.sqrt(1 - B**2)
    
    cos_theta_prime = (np.cos(theta) - B) / (1 - B*np.cos(theta))
    sin_theta_prime = np.sin(theta) / (Gamma*(1 - B*np.cos(theta)))
    theta_prime = atan2(sin_theta_prime, cos_theta_prime) * u.rad
    
    # находим r'
    gamma = sin_theta_prime / np.sin(theta - theta_prime)
    
    r_prime_cart = (e/v + gamma*r) / np.sqrt(1 + gamma**2 + 2*gamma*(r.dot(e/v)))
    r_prime_coord = SkyCoord(r_prime_cart, frame='icrs')
    
    ra_ab = r_prime_coord.ra.to(u.rad).value
    dec_ab = r_prime_coord.dec.to(u.rad).value
    
    return ra_ab, dec_ab

In [143]:
print(Time(datetime.utcnow(), scale='tt').copy(format='jd'))
JD = float(Time(datetime.utcnow()).copy(format='jd').value)

obj = Horizons(id='3', id_type='majorbody', location='500@0', epochs=JD)
print(obj)
ev= obj.vectors(refplane='earth')
# 'targetname','datetime_jd','datetime_str','x','y','z','vx','vy','vz','lighttime','range','range_rate'

print(ev['x'][0],ev['y'][0],ev['z'][0],ev['vx'][0],ev['vy'][0],ev['vz'][0])

2461181.1627714457
JPLHorizons instance "3"; location=500@0, epochs=[2461181.162771457], id_type=None


/usr/local/lib/python3.7/dist-packages/urllib3/connectionpool.py:1020: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ssd.jpl.nasa.gov'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,


-0.5189580804258347 -0.8031942657896984 -0.348044938525287 0.01451833473873776 -0.008120340166310299 -0.0035202583875651


In [144]:
ra,dec = radians(45.0),radians(40.0)
# ra,dec = radians(0.0),radians(0.0)
JDs = np.linspace(JD,JD+365.25,50)
obj = Horizons(id='3', id_type='majorbody', location='500@0', epochs=JDs)
EV= obj.vectors(refplane='earth')

x,y = np.zeros(np.size(JDs)),np.zeros(np.size(JDs))
for k, ev in enumerate(EV):
    ra_ab,dec_ab = aberration(ra,dec,ev['vx'],ev['vy'],ev['vz'])
    tx,ty = tangFromRADE(ra_ab,dec_ab,ra,dec)
    x[k],y[k] = 3600*degrees(tx),3600*degrees(ty)

/usr/local/lib/python3.7/dist-packages/urllib3/connectionpool.py:1020: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ssd.jpl.nasa.gov'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,


In [145]:
import matplotlib.pyplot as plt
%matplotlib widget

fig, ax = plt.subplots(figsize=(3, 2), dpi=300, tight_layout=True)
ax.set_aspect(aspect=1)
plt.scatter(x, y, c=JDs, s=0.2)
plt.plot(x, y)
plt.xlim(-30,30)
plt.ylim(-30,30)
ax.xaxis.set_tick_params(labelsize=3)
ax.yaxis.set_tick_params(labelsize=3)
plt.xlabel('$\\Delta \\alpha$, arcsec', fontsize = 4)
plt.ylabel('$\\Delta \\delta$, arcsec', fontsize = 4)
plt.tight_layout()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …